<a href="https://colab.research.google.com/github/Aniruddha-Ray/Atlas-Guided-Medical-Image-Segmentation/blob/main/Atlas_Guided_Medical_Image_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [10]:
%pip install -q monai nibabel einops


Note: you may need to restart the kernel to use updated packages.


In [1]:
import monai
print(monai.__version__)

1.6.0


## Shared Constants & Imports

In [2]:
import os, glob, random, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
import monai.transforms as mt
from monai.data import CacheDataset, DataLoader

# ── Fixed across ALL models for a fair comparison ──
IMG_SIZE    = (96, 96, 96)
NUM_CLASSES = 13          # AMOS22: background + 12 organs
BATCH_SIZE  = 1           # 3-D volumes are large
NUM_WORKERS = 0           # avoids OOM-killed worker processes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# AMOS22 organ label map (class 0 = background, excluded from Dice)
ORGAN_NAMES = [
    "Organ 1", "Organ 2", "Organ 3", "Organ 4", "Organ 5",
    "Organ 6", "Organ 7", "Organ 8", "Organ 9", "Organ 10", "Organ 11", "Organ 12"
]


Device: cuda


## Data Preparation

All comparison models share **identical** data loaders — image `(B,1,96,96,96)` float and label `(B,1,96,96,96)` long (integer class index per voxel). No distance maps or probability maps are needed here.

In [3]:
import monai
monai.data.set_track_meta(False)   # halves intermediate memory during resampling

data_dir = "C:\\Users\\Asus\\OneDrive\\Desktop\\Atlas Based Medical Segmentation\\Dataset060_Merged_Def" 

image_files = sorted(glob.glob(os.path.join(data_dir, "*_0000.nii.gz")))
label_files_set = set(glob.glob(os.path.join(data_dir, "*.nii.gz")))

# ── Pair image with its label, then shuffle PAIRS together ──
random.seed(42)
data_dicts = []
for img_path in image_files:
    lbl_path = os.path.join(data_dir,
                  os.path.basename(img_path).replace("_0000.nii.gz", ".nii.gz"))
    if lbl_path in label_files_set:
        data_dicts.append({"image": img_path, "label": lbl_path})

random.shuffle(data_dicts)           # shuffle PAIRS so image-label correspondence is kept

split = int(len(data_dicts) * 0.8)
train_files = data_dicts[:split]
val_files   = data_dicts[split:]

print(f"Total paired samples : {len(data_dicts)}")
print(f"Training samples     : {len(train_files)}")
print(f"Validation samples   : {len(val_files)}")
print("\nSample pair check:")
for d in train_files[:2]:
    print(f"  {os.path.basename(d['image'])}  →  {os.path.basename(d['label'])}")


Total paired samples : 680
Training samples     : 544
Validation samples   : 136

Sample pair check:
  amos_0296_0000.nii.gz  →  amos_0296.nii.gz
  amos_0200_0000.nii.gz  →  amos_0200.nii.gz


In [12]:
import monai
IMG_SIZE = (96, 96, 96)
NUM_CLASSES = 13

# image_only=False preserves NIfTI metadata (affine, original_channel_dim)
# which EnsureChannelFirstd needs to know there is no channel dim.
# Without it EnsureChannelFirstd raises ValueError on local MONAI installs.

train_transforms = mt.Compose([
    mt.LoadImaged(keys=["image", "label"], image_only=False),  # ← this is the only change
    mt.EnsureChannelFirstd(keys=["image", "label"], channel_dim="no_channel"),
    mt.Orientationd(keys=["image", "label"], axcodes="RAS"),
    mt.CropForegroundd(keys=["image", "label"], source_key="image"),
    mt.Spacingd(keys=["image", "label"],
                pixdim=(1.5, 1.5, 2.0),
                mode=("bilinear", "nearest")),
    mt.ScaleIntensityRanged(keys=["image"],
                            a_min=-1000, a_max=1000,
                            b_min=0.0, b_max=1.0, clip=True),
    mt.Resized(keys=["image", "label"],
               spatial_size=IMG_SIZE,
               mode=("area", "nearest")),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    mt.RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    mt.RandRotated(keys=["image", "label"],
                   range_x=(-0.26, 0.26),
                   range_y=(-0.26, 0.26),
                   range_z=(-0.26, 0.26),
                   prob=0.5, mode=("bilinear", "nearest")),
    mt.CastToTyped(keys=["image"], dtype=torch.float32),
    mt.CastToTyped(keys=["label"], dtype=torch.long),
])

val_transforms = mt.Compose([
    mt.LoadImaged(keys=["image", "label"], image_only=False),
    mt.EnsureChannelFirstd(keys=["image", "label"],channel_dim="no_channel"),
    mt.Orientationd(keys=["image", "label"], axcodes="RAS"),
    mt.CropForegroundd(keys=["image", "label"], source_key="image"),
    mt.Spacingd(keys=["image", "label"],
                pixdim=(1.5, 1.5, 2.0),
                mode=("bilinear", "nearest")),
    mt.ScaleIntensityRanged(keys=["image"],
                            a_min=-1000, a_max=1000,
                            b_min=0.0, b_max=1.0, clip=True),
    mt.Resized(keys=["image", "label"],
               spatial_size=IMG_SIZE,
               mode=("area", "nearest")),
    mt.CastToTyped(keys=["image"], dtype=torch.float32),
    mt.CastToTyped(keys=["label"], dtype=torch.long),
])

train_ds = CacheDataset(data=train_files, transform=train_transforms, cache_rate=0.0)
val_ds   = CacheDataset(data=val_files,   transform=val_transforms,   cache_rate=0.0)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=0)

print(f"MONAI version  : {monai.__version__}")
print(f"Train batches  : {len(train_loader)}")
print(f"Val   batches  : {len(val_loader)}")


MONAI version  : 1.6.0
Train batches  : 544
Val   batches  : 136


In [9]:
import monai.transforms as mt
import torch
import nibabel as nib
import numpy as np

# load one raw file directly
img = nib.load(train_files[0]["image"]).get_fdata()
lbl = nib.load(train_files[0]["label"]).get_fdata()

print("image numpy shape:", img.shape)
print("label numpy shape:", lbl.shape)

# now try just LoadImaged + EnsureChannelFirstd with strict_check=False
# so it warns instead of raises, revealing the actual message
test_transform = mt.Compose([
    mt.LoadImaged(keys=["image", "label"], image_only=False),
    mt.EnsureChannelFirstd(keys=["image", "label"], strict_check=False),
])

try:
    out = test_transform(train_files[0])
    print("image tensor shape:", out["image"].shape)
    print("label tensor shape:", out["label"].shape)
except Exception as e:
    print("ERROR:", e)

image numpy shape: (512, 372, 187)
label numpy shape: (512, 372, 187)
image tensor shape: torch.Size([512, 372, 187])
label tensor shape: torch.Size([512, 372, 187])


c:\Users\Asus\anaconda3\Lib\site-packages\monai\transforms\utility\array.py:206: UserWarning: Metadata not available and channel_dim=None, EnsureChannelFirst is not in use.
  warnings.warn(msg)


In [10]:
test_transform = mt.Compose([
    mt.LoadImaged(keys=["image", "label"], image_only=False),
    mt.EnsureChannelFirstd(keys=["image", "label"], channel_dim="no_channel"),
])

out = test_transform(train_files[0])
print("image:", out["image"].shape)   # expected: (1, 512, 372, 187)
print("label:", out["label"].shape)   # expected: (1, 512, 372, 187)

image: torch.Size([1, 512, 372, 187])
label: torch.Size([1, 512, 372, 187])


## Shared Validation Helper

Single function reused by every model's validation loop so the Dice computation is identical and results are comparable.

In [13]:
def validate(model, val_loader, dice_metric, device, NUM_CLASSES, epoch, num_epochs, model_name):
    """
    Runs one validation pass.
    - Discretises predictions correctly: softmax → argmax → F.one_hot → permute
    - Converts integer labels to one-hot for DiceMetric
    - Returns mean foreground Dice and per-class Dice tensor
    """
    model.eval()
    dice_metric.reset()

    with torch.no_grad():
        for val_batch in val_loader:
            val_images = val_batch["image"].to(device)   # (B,1,96,96,96)
            val_labels = val_batch["label"].to(device)   # (B,1,96,96,96) long

            val_logits = model(val_images)               # (B,13,96,96,96) raw logits

            # Discretise predictions
            val_preds_discrete = F.one_hot(
                F.softmax(val_logits, dim=1).argmax(dim=1).long(),
                num_classes=NUM_CLASSES
            ).permute(0, 4, 1, 2, 3).float()            # (B,13,96,96,96)

            # Convert integer labels to one-hot
            val_labels_onehot = F.one_hot(
                val_labels.squeeze(1).long(),
                num_classes=NUM_CLASSES
            ).permute(0, 4, 1, 2, 3).float()            # (B,13,96,96,96)

            dice_metric(y_pred=val_preds_discrete, y=val_labels_onehot)

    per_class_dice, not_nans = dice_metric.aggregate()
    mean_dice = per_class_dice[not_nans.bool()].mean().item()

    print(f"[{model_name}] Epoch [{epoch+1}/{num_epochs}] Val Mean Dice: {mean_dice:.4f}")
    print(f"  {'Organ':<25} {'Dice':>6}")
    print("  " + "-"*32)
    for name, score, present in zip(ORGAN_NAMES, per_class_dice, not_nans):
        tag = f"{score:.4f}" if present else "N/A"
        print(f"  {name:<25} {tag:>6}")

    return mean_dice, per_class_dice

print("validate() helper ready.")


validate() helper ready.


---
## Model 1 — UNETR (Baseline)

Stock MONAI UNETR with ViT encoder and CNN decoder. No atlas guidance, no cross-attention. This is the direct baseline for the Atlas-Guided ViT-UNETR.

In [6]:
from monai.networks.nets import UNETR

unetr_model = UNETR(
    in_channels=1,
    out_channels=NUM_CLASSES,        # 13 classes
    img_size=IMG_SIZE,               # (96,96,96)
    feature_size=16,
    hidden_size=768,
    mlp_dim=3072,
    num_heads=12,
    proj_type="perceptron",
    norm_name="instance",
    res_block=True,
).to(device)

n_params = sum(p.numel() for p in unetr_model.parameters())
print(f"UNETR parameters: {n_params:,}")


UNETR parameters: 92,802,461


In [ ]:
# Re-instantiate criterion and optimizer for the new AtlasGuidedViTUNETR model
criterion = DiceCELoss(
    to_onehot_y=False, # Labels are already one-hot from transforms
    softmax=True
)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-5
)

print("Loss function and Optimizer re-initialized for AtlasGuidedViTUNETR.")

dice_metric = DiceMetric(
    include_background=False,
    reduction="mean_batch",
    get_not_nans= True
)

ORGAN_NAMES = [
    "Organ 1", "Oragan 2", "Organ 3", "Organ 4", "Organ 5", "Organ 6",
    "Organ 7", "Organ 8", "Organ 9", "Organ 10", "Organ 11", "Organ 12"
]

In [14]:
num_epochs = 5

print("Training UNETR...")
for epoch in range(num_epochs):

    unetr_model.train()
    epoch_loss = 0.0

    for batch in train_loader:
        images = batch["image"].to(device)    # (B,1,96,96,96)
        labels = batch["label"].to(device)    # (B,1,96,96,96) long

        unetr_optimizer.zero_grad()
        outputs = unetr_model(images)         # (B,13,96,96,96) raw logits
        loss = unetr_criterion(outputs, labels)
        loss.backward()
        unetr_optimizer.step()
        epoch_loss += loss.item()

    print(f"[UNETR] Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss/len(train_loader):.4f}")

    validate(unetr_model, val_loader, unetr_dice_metric,
             device, NUM_CLASSES, epoch, num_epochs, "UNETR")

torch.save({"state_dict": unetr_model.state_dict()}, "unetr_baseline.pth")
print("UNETR saved.")


Training UNETR...


c:\Users\Asus\anaconda3\Lib\site-packages\monai\transforms\spatial\array.py:656: UserWarning: `data_array` is not of type `MetaTensor, assuming affine to be identity.
  warnings.warn("`data_array` is not of type `MetaTensor, assuming affine to be identity.")
c:\Users\Asus\anaconda3\Lib\site-packages\monai\transforms\spatial\array.py:500: UserWarning: `data_array` is not of type MetaTensor, assuming affine to be identity.
  warnings.warn("`data_array` is not of type MetaTensor, assuming affine to be identity.")


[UNETR] Epoch [1/5] Loss: 2.9193
[UNETR] Epoch [1/5] Val Mean Dice: 0.0436
  Organ                       Dice
  --------------------------------
  Organ 1                   0.0000
  Organ 2                   0.0000
  Organ 3                   0.0421
  Organ 4                   0.0000
  Organ 5                   0.0001
  Organ 6                   0.4793
  Organ 7                   0.0000
  Organ 8                   0.0000
  Organ 9                   0.0016
  Organ 10                  0.0001
  Organ 11                  0.0001
  Organ 12                  0.0002
[UNETR] Epoch [2/5] Loss: 2.2401
[UNETR] Epoch [2/5] Val Mean Dice: 0.0490
  Organ                       Dice
  --------------------------------
  Organ 1                   0.0000
  Organ 2                   0.0000
  Organ 3                   0.0000
  Organ 4                   0.0000
  Organ 5                   0.0000
  Organ 6                   0.5879
  Organ 7                   0.0000
  Organ 8                   0.0000
  Organ 9 

---
## Model 2 — UNet 3D

MONAI's 3D UNet with skip connections. A classical CNN segmentation baseline. Note the original notebook used `spatial_dims=2` and set it up as an image **classifier** — corrected here to 3D volumetric **segmentation**.

In [15]:
from monai.networks.nets import UNet

unet_model = UNet(
    spatial_dims=3,                       # 3D volumetric segmentation
    in_channels=1,
    out_channels=NUM_CLASSES,             # 13 classes
    channels=(32, 64, 128, 256, 512),     # encoder feature map sizes
    strides=(2, 2, 2, 2),                 # downsampling at each stage
    num_res_units=2,                      # residual units per block
    norm="instance",
).to(device)

n_params = sum(p.numel() for p in unet_model.parameters())
print(f"UNet3D parameters: {n_params:,}")


UNet3D parameters: 19,240,414


In [16]:
unet_criterion = DiceCELoss(
    to_onehot_y=True,
    softmax=True,
)

unet_optimizer = torch.optim.AdamW(
    unet_model.parameters(), lr=1e-4, weight_decay=1e-5
)

unet_dice_metric = DiceMetric(
    include_background=False, reduction="mean_batch", get_not_nans=True
)


In [17]:
num_epochs = 5

print("Training UNet3D...")
for epoch in range(num_epochs):

    unet_model.train()
    epoch_loss = 0.0

    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        unet_optimizer.zero_grad()
        outputs = unet_model(images)
        loss = unet_criterion(outputs, labels)
        loss.backward()
        unet_optimizer.step()
        epoch_loss += loss.item()

    print(f"[UNet3D] Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss/len(train_loader):.4f}")

    validate(unet_model, val_loader, unet_dice_metric,
             device, NUM_CLASSES, epoch, num_epochs, "UNet3D")

torch.save({"state_dict": unet_model.state_dict()}, "unet3d.pth")
print("UNet3D saved.")


Training UNet3D...


c:\Users\Asus\anaconda3\Lib\site-packages\monai\transforms\spatial\array.py:656: UserWarning: `data_array` is not of type `MetaTensor, assuming affine to be identity.
  warnings.warn("`data_array` is not of type `MetaTensor, assuming affine to be identity.")
c:\Users\Asus\anaconda3\Lib\site-packages\monai\transforms\spatial\array.py:500: UserWarning: `data_array` is not of type MetaTensor, assuming affine to be identity.
  warnings.warn("`data_array` is not of type MetaTensor, assuming affine to be identity.")


[UNet3D] Epoch [1/5] Loss: 1.4943
[UNet3D] Epoch [1/5] Val Mean Dice: 0.0610
  Organ                       Dice
  --------------------------------
  Organ 1                   0.0000
  Organ 2                   0.0000
  Organ 3                   0.0000
  Organ 4                   0.0000
  Organ 5                   0.0000
  Organ 6                   0.7315
  Organ 7                   0.0000
  Organ 8                   0.0000
  Organ 9                   0.0000
  Organ 10                  0.0000
  Organ 11                  0.0000
  Organ 12                  0.0000
[UNet3D] Epoch [2/5] Loss: 1.0212
[UNet3D] Epoch [2/5] Val Mean Dice: 0.1407
  Organ                       Dice
  --------------------------------
  Organ 1                   0.2657
  Organ 2                   0.1612
  Organ 3                   0.0928
  Organ 4                   0.0000
  Organ 5                   0.0000
  Organ 6                   0.7751
  Organ 7                   0.3914
  Organ 8                   0.0006
  Orga